In [8]:
pip install langchain

In [10]:
!pip install -q langchain-community docx2txt

In [9]:
from langchain_community.document_loaders import Docx2txtLoader

/tmp/ipykernel_3982/1675806912.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import Docx2txtLoader


In [11]:
from langchain_community.document_loaders import Docx2txtLoader
import os

from langchain_community.document_loaders import Docx2txtLoader
import os

# Annual report file paths
files = {
    "2023": "/content/Data/2023_Annual_Report.docx",
    "2024": "/content/Data/2024_Annual_Report.docx",
    "2025": "/content/Data/2025_AnnualReport.docx"
}

all_documents = []

# Load each annual report
for year, file_path in files.items():

    if not os.path.exists(file_path):
        print(f"File not found: {file_path}")
        continue

    loader = Docx2txtLoader(file_path)
    documents = loader.load()

    # Add metadata
    for doc in documents:
        doc.metadata["year"] = year
        doc.metadata["source"] = os.path.basename(file_path)
        doc.metadata["document_type"] = "Annual Report"

    all_documents.extend(documents)

print("Total reports loaded:", len(all_documents))
print("Total characters:", sum(len(doc.page_content) for doc in all_documents))

# Preview the first report
if all_documents:
    print("\nFirst report metadata:")
    print(all_documents[0].metadata)

    print("\nFirst 500 characters:")
    print(all_documents[0].page_content[:500])

Total reports loaded: 3
Total characters: 842564

First report metadata:
{'source': '2023_Annual_Report.docx', 'year': '2023', 'document_type': 'Annual Report'}

First 500 characters:
Dear shareholders, colleagues, customers, and partners, 

We are living through a time of historic challenge and opportunity. As I write this, the world faces ongoing economic, social, and geopolitical volatility. At the same time, we have entered a new age of AI that will fundamentally transform productivity for every individual, organization, and industry on earth, and help us address some of our most pressing challenges. 

This next generation of AI will reshape every software category and ev


In [12]:
print(all_documents[0].page_content[:2000])

Dear shareholders, colleagues, customers, and partners, 

We are living through a time of historic challenge and opportunity. As I write this, the world faces ongoing economic, social, and geopolitical volatility. At the same time, we have entered a new age of AI that will fundamentally transform productivity for every individual, organization, and industry on earth, and help us address some of our most pressing challenges. 

This next generation of AI will reshape every software category and every business, including our own. Forty-eight years after its founding, Microsoft remains a consequential company because time and time again—from PC/Server, to Web/Internet, to Cloud/Mobile—we have adapted to technological paradigm shifts. Today, we are doing so once again, as we lead this new era. 

Amid this transformation, our mission to empower every person and every organization on the planet to achieve more remains constant. As a company, we believe we can be the democratizing force for th

# chunking

In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [14]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

In [15]:
chunks = text_splitter.split_documents(
    all_documents
)

print("Original documents:", len(all_documents))
print("Total chunks:", len(chunks))

Original documents: 3
Total chunks: 1143


In [16]:
print(chunks[0].metadata)

{'source': '2023_Annual_Report.docx', 'year': '2023', 'document_type': 'Annual Report'}


In [17]:
print("First chunk characters:", len(chunks[0].page_content))

First chunk characters: 799


# embedding

In [18]:
!pip install -q langchain-huggingface sentence-transformers

In [19]:
from langchain_huggingface import HuggingFaceEmbeddings

In [20]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully


In [21]:
text = chunks[0].page_content

vector = embeddings.embed_query(text)

In [22]:
print("Vector length:", len(vector))
print("First 10 values:", vector[:10])

Vector length: 384
First 10 values: [-0.04844231903553009, -0.02738860249519348, 0.055798452347517014, -0.0007778463768772781, 0.06601578742265701, 0.0169635359197855, 0.014981215819716454, -0.03623655065894127, 0.01428846176713705, -0.01956944353878498]


# vector db

In [23]:
%pip install chromadb

In [24]:
import chromadb

# Create a persistent ChromaDB database
client = chromadb.PersistentClient(
    path="/content/chroma_financial_v3"
)

# Create a collection for annual reports
collection = client.get_or_create_collection(
    name="financial_reports_v3",
    metadata={"hnsw:space": "cosine"}
)

print("ChromaDB collection created successfully!")

ChromaDB collection created successfully!


In [26]:
# Prepare document text
texts = [chunk.page_content for chunk in chunks]

# Prepare metadata
metadatas = [
    {
        "year": str(chunk.metadata.get("year", "Unknown")),
        "source": str(chunk.metadata.get("source", "Unknown")),
        "document_type": str(
            chunk.metadata.get("document_type", "Annual Report")
        )
    }
    for chunk in chunks
]

# Generate embeddings for all chunks
vectors = embeddings.embed_documents(texts)

print("Chunks:", len(texts))
print("Embeddings:", len(vectors))
print("Vector dimensions:", len(vectors[0]))

Chunks: 1143
Embeddings: 1143
Vector dimensions: 384


In [27]:
collection.upsert(
    ids=[f"report_chunk_{i}" for i in range(len(texts))],
    documents=texts,
    metadatas=metadatas,
    embeddings=vectors
)

print("Stored chunks:", collection.count())

Stored chunks: 1143


In [28]:
results = collection.get(
    limit=2,
    include=["documents", "metadatas"]
)

for i in range(len(results["ids"])):
    print("\nID:", results["ids"][i])
    print("Metadata:", results["metadatas"][i])
    print("Text:", results["documents"][i][:200])


ID: report_chunk_0
Metadata: {'year': '2023', 'source': '2023_Annual_Report.docx', 'document_type': 'Annual Report'}
Text: Dear shareholders, colleagues, customers, and partners, 

We are living through a time of historic challenge and opportunity. As I write this, the world faces ongoing economic, social, and geopolitica

ID: report_chunk_1
Metadata: {'year': '2023', 'source': '2023_Annual_Report.docx', 'document_type': 'Annual Report'}
Text: Amid this transformation, our mission to empower every person and every organization on the planet to achieve more remains constant. As a company, we believe we can be the democratizing force for this


In [29]:
def retrieve_documents(question, top_k=5, year=None):

    # Convert the question into an embedding
    query_vector = embeddings.embed_query(question)

    # Search ChromaDB
    query_args = {
        "query_embeddings": [query_vector],
        "n_results": top_k,
        "include": ["documents", "metadatas", "distances"]
    }

    # Optional filter by fiscal/report year
    if year is not None:
        query_args["where"] = {"year": str(year)}

    results = collection.query(**query_args)

    retrieved_docs = []

    for i, document in enumerate(results["documents"][0]):
        retrieved_docs.append({
            "text": document,
            "metadata": results["metadatas"][0][i],
            "distance": results["distances"][0][i]
        })

    return retrieved_docs

In [30]:
question = "What was Microsoft's total revenue in fiscal year 2025?"

retrieved_docs = retrieve_documents(question, top_k=5)

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"\n--- Result {i} ---")
    print("Year:", doc["metadata"]["year"])
    print("Source:", doc["metadata"]["source"])
    print("Distance:", round(doc["distance"], 4))
    print("Text:", doc["text"][:500])


--- Result 1 ---
Year: 2025
Source: 2025_AnnualReport.docx
Distance: 0.2782
Text: Operating Income

  

$

128,528

 

  

$

109,433

 

  

 

17%

 

 

  

 

 



  

 

 



  

 

 

 

Reportable Segments 

Fiscal Year 2025 Compared with Fiscal Year 2024 

Productivity and Business Processes 

Revenue increased $14.0 billion or 13%. 

•	Microsoft 365 Commercial products and cloud services revenue increased $10.8 billion or 14%. Microsoft 365 Commercial cloud revenue grew 15% with Microsoft 365 Commercial seat growth of 6% driven by small and medium businesses and fron

--- Result 2 ---
Year: 2024
Source: 2024_Annual_Report.docx
Distance: 0.3039
Text: Highlights from fiscal year 2024 compared with fiscal year 2023 included: 

•	Microsoft Cloud revenue increased 23% to $137.4 billion. 

•	Office Commercial products and cloud services revenue increased 14% driven by Office 365 Commercial growth of 16%. 

•	Office Consumer products and cloud services revenue increased 4% and Micro

In [31]:
question = "What was Microsoft's total company revenue for fiscal year 2025 in millions of dollars?"

retrieved_docs = retrieve_documents(question, top_k=5, year=2025)

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"\n--- Result {i} ---")
    print("Source:", doc["metadata"]["source"])
    print("Distance:", round(doc["distance"], 4))
    print(doc["text"][:800])


--- Result 1 ---
Source: 2025_AnnualReport.docx
Distance: 0.3054
Operating Income

  

$

128,528

 

  

$

109,433

 

  

 

17%

 

 

  

 

 



  

 

 



  

 

 

 

Reportable Segments 

Fiscal Year 2025 Compared with Fiscal Year 2024 

Productivity and Business Processes 

Revenue increased $14.0 billion or 13%. 

•	Microsoft 365 Commercial products and cloud services revenue increased $10.8 billion or 14%. Microsoft 365 Commercial cloud revenue grew 15% with Microsoft 365 Commercial seat growth of 6% driven by small and medium businesses and frontline worker offerings, as well as growth in revenue per user. Microsoft 365 Commercial products revenue grew 7% driven by the Windows Commercial on-premises components of Microsoft 365 suite sales and an increase in Office transactional purchasing with the launch of Office 2024.

--- Result 2 ---
Source: 2025_AnnualReport.docx
Distance: 0.3071
137,178

 

  

 

120,418

 

  

 

105,171

 

 



  

 

 



  

 

 



Total

 

In [43]:
%pip install -q google-genai
%pip install -q langchain-google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 802.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.0/268.0 kB 19.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.61.0 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.1 which is incompatible.


In [44]:
from google import genai

API_KEY = open("api.txt", "r").read().strip()

client = genai.Client(api_key=API_KEY)

print("Gemini client created successfully!")

Gemini client created successfully!


In [54]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    google_api_key=API_KEY,
    temperature=0
)

print("Gemini model configured successfully!")

Gemini model configured successfully!


In [55]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = PromptTemplate(
    template="""
You are a financial report question-answering assistant.

Answer using ONLY the provided context.
- Do not guess financial figures.
- Give a direct answer.
- Mention the source year when available.
- If the answer is missing, say:
"I could not find this information in the provided reports."

Context:
{context}

Question:
{question}

Answer:
""",
    input_variables=["context", "question"]
)

parser = StrOutputParser()

# LangChain Runnable chain
chain = prompt | llm | parser

print("V3 Runnable chain created successfully!")

V3 Runnable chain created successfully!


In [56]:
def ask_report_v3(question, k=5, year=None):

    # 1. Convert question into an embedding
    query_vector = embeddings.embed_query(question)

    # 2. Search ChromaDB
    search_args = {
        "query_embeddings": [query_vector],
        "n_results": min(k, collection.count()),
        "include": ["documents", "metadatas", "distances"]
    }

    # Optional filter by report year
    if year is not None:
        search_args["where"] = {"year": str(year)}

    results = collection.query(**search_args)

    documents = results["documents"][0]
    metadatas = results["metadatas"][0]

    if not documents:
        return "No relevant report passages found.", []

    # 3. Build context
    context = "\n\n".join(
        f"Source: {metadata.get('source', 'Unknown')}\n"
        f"Year: {metadata.get('year', 'Unknown')}\n"
        f"{document}"
        for document, metadata in zip(documents, metadatas)
    )

    # 4. Generate answer using the Runnable chain
    answer = chain.invoke({
        "context": context,
        "question": question
    })

    # 5. Return source information
    sources = [
        {
            "source": metadata.get("source"),
            "year": metadata.get("year"),
            "distance": distance
        }
        for metadata, distance in zip(
            metadatas, results["distances"][0]
        )
    ]

    return answer, sources

In [57]:
question = "What was Microsoft's total revenue in fiscal year 2025?"

answer, sources = ask_report_v3(
    question,
    k=5,
    year=2025
)

print("QUESTION:", question)
print("\nANSWER:\n", answer)

print("\nSOURCES:")
for source in sources:
    print(
        source["source"],
        "| Year:", source["year"],
        "| Distance:", round(source["distance"], 4)
    )

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


QUESTION: What was Microsoft's total revenue in fiscal year 2025?

ANSWER:
 Based on the 2025 annual report, Microsoft's total revenue in fiscal year 2025 was $281,724 million (or $281.724 billion).

SOURCES:
2025_AnnualReport.docx | Year: 2025 | Distance: 0.2782
2025_AnnualReport.docx | Year: 2025 | Distance: 0.3203
2025_AnnualReport.docx | Year: 2025 | Distance: 0.3299
2025_AnnualReport.docx | Year: 2025 | Distance: 0.3335
2025_AnnualReport.docx | Year: 2025 | Distance: 0.3366


In [58]:
print(
    ask_report_v3(
        "What was Microsoft's total company revenue in fiscal year 2025?",
        year=2025
    )
)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


("Based on the 2025 Annual Report, Microsoft's total company revenue in fiscal year 2025 was $281,724 million (or $281.724 billion).", [{'source': '2025_AnnualReport.docx', 'year': '2025', 'distance': 0.29012954235076904}, {'source': '2025_AnnualReport.docx', 'year': '2025', 'distance': 0.31785285472869873}, {'source': '2025_AnnualReport.docx', 'year': '2025', 'distance': 0.34407103061676025}, {'source': '2025_AnnualReport.docx', 'year': '2025', 'distance': 0.34491586685180664}, {'source': '2025_AnnualReport.docx', 'year': '2025', 'distance': 0.35053062438964844}])


In [59]:
print(ask_report_v3("What was Microsoft's total revenue in fiscal year 2023?", year=2023))

print(ask_report_v3("What was Microsoft's total revenue in fiscal year 2024?", year=2024))

print(ask_report_v3("What was Microsoft's net income in fiscal year 2025?", year=2025))

print(ask_report_v3("What was Microsoft's operating income in fiscal year 2025?", year=2025))

print(ask_report_v3("How much revenue did Microsoft Cloud generate in fiscal year 2025?", year=2025))

print(ask_report_v3("What were Microsoft's main reportable business segments in fiscal year 2025?", year=2025))

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


("Based on the 2023 Annual Report, Microsoft's total revenue in fiscal year 2023 was $211,915 (in millions, as indicated by the total figures in the provided financial tables).", [{'source': '2023_Annual_Report.docx', 'year': '2023', 'distance': 0.33190762996673584}, {'source': '2023_Annual_Report.docx', 'year': '2023', 'distance': 0.3428384065628052}, {'source': '2023_Annual_Report.docx', 'year': '2023', 'distance': 0.3468090295791626}, {'source': '2023_Annual_Report.docx', 'year': '2023', 'distance': 0.3471999764442444}, {'source': '2023_Annual_Report.docx', 'year': '2023', 'distance': 0.35487544536590576}])


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


('I could not find this information in the provided reports.', [{'source': '2024_Annual_Report.docx', 'year': '2024', 'distance': 0.29857778549194336}, {'source': '2024_Annual_Report.docx', 'year': '2024', 'distance': 0.3153644800186157}, {'source': '2024_Annual_Report.docx', 'year': '2024', 'distance': 0.33700263500213623}, {'source': '2024_Annual_Report.docx', 'year': '2024', 'distance': 0.3389878273010254}, {'source': '2024_Annual_Report.docx', 'year': '2024', 'distance': 0.35015493631362915}])


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


...

("Based on the 2025 Annual Report (Source: 2025_AnnualReport.docx), Microsoft's operating income in fiscal year 2025 was $128,528 million.", [{'source': '2025_AnnualReport.docx', 'year': '2025', 'distance': 0.29734915494918823}, {'source': '2025_AnnualReport.docx', 'year': '2025', 'distance': 0.33456820249557495}, {'source': '2025_AnnualReport.docx', 'year': '2025', 'distance': 0.38032060861587524}, {'source': '2025_AnnualReport.docx', 'year': '2025', 'distance': 0.381489634513855}, {'source': '2025_AnnualReport.docx', 'year': '2025', 'distance': 0.3931829333305359}])


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


('Based on the 2025 Annual Report, Microsoft Cloud revenue increased 23% to $168.9 billion in fiscal year 2025.', [{'source': '2025_AnnualReport.docx', 'year': '2025', 'distance': 0.19174915552139282}, {'source': '2025_AnnualReport.docx', 'year': '2025', 'distance': 0.23332488536834717}, {'source': '2025_AnnualReport.docx', 'year': '2025', 'distance': 0.23624932765960693}, {'source': '2025_AnnualReport.docx', 'year': '2025', 'distance': 0.24424076080322266}, {'source': '2025_AnnualReport.docx', 'year': '2025', 'distance': 0.2522014379501343}])


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


('Based on the provided 2025 Annual Report, the segments were aligned to bring the commercial components of Microsoft 365 together in the **Productivity and Business Processes** segment. (Note: The provided text details the Productivity and Business Processes segment, but does not list all main reportable business segments by name beyond this one.)', [{'source': '2025_AnnualReport.docx', 'year': '2025', 'distance': 0.2414531707763672}, {'source': '2025_AnnualReport.docx', 'year': '2025', 'distance': 0.25775146484375}, {'source': '2025_AnnualReport.docx', 'year': '2025', 'distance': 0.3199880123138428}, {'source': '2025_AnnualReport.docx', 'year': '2025', 'distance': 0.37606269121170044}, {'source': '2025_AnnualReport.docx', 'year': '2025', 'distance': 0.3800312876701355}])


In [60]:
print(
    ask_report_v3(
        "What was Microsoft's total company revenue in fiscal year 2025?",
        year=2025
    )[0]
)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


In fiscal year 2025, Microsoft's total company revenue was $281,724 million (or $281.724 billion).


In [61]:
result = ask_report_v3(
    "What was Microsoft's total company revenue in fiscal year 2025?",
    k=5,
    year=2025
)

print("ANSWER:")
print(result[0])

print("\nRETRIEVED SOURCES:")
for source in result[1]:
    print(
        "Year:", source["year"],
        "| File:", source["source"],
        "| Distance:", round(source["distance"], 4)
    )

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


ANSWER:
Based on the 2025 Annual Report, Microsoft's total company revenue in fiscal year 2025 was $281,724 million (or $281.724 billion).

RETRIEVED SOURCES:
Year: 2025 | File: 2025_AnnualReport.docx | Distance: 0.2901
Year: 2025 | File: 2025_AnnualReport.docx | Distance: 0.3179
Year: 2025 | File: 2025_AnnualReport.docx | Distance: 0.3441
Year: 2025 | File: 2025_AnnualReport.docx | Distance: 0.3449
Year: 2025 | File: 2025_AnnualReport.docx | Distance: 0.3505


In [62]:
print(ask_report_v3(
    "What was Microsoft's net income in fiscal year 2025?",
    k=5,
    year=2025
)[0])

print(ask_report_v3(
    "What was Microsoft's total revenue in fiscal year 2024?",
    k=5,
    year=2024
)[0])

print(ask_report_v3(
    "How much revenue did Microsoft Cloud generate in fiscal year 2025?",
    k=5,
    year=2025
)[0])

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Based on the 2025 Annual Report (Source: 2025_AnnualReport.docx), Microsoft's net income in fiscal year 2025 was $101,832 million.


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


I could not find this information in the provided reports.


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


I could not find this information in the provided reports.


In [63]:
print(
    ask_report_v3(
        "What was Microsoft's total revenue in fiscal year 2024?",
        k=5,
        year=2024
    )[0]
)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


I could not find this information in the provided reports.


In [64]:
results = retrieve_documents(
    "Microsoft total revenue 2024 245122 million",
    top_k=3,
    year=2024
)

for item in results:
    print(item["text"][:1000])
    print("Distance:", round(item["distance"], 4))
    print("-" * 50)

233	

 

 

 

Total

		$	 245,122	

		$	 211,915	

		$	 198,270	

 

 

 

 

We have recast certain prior period amounts to conform to the way we internally manage and monitor our business. 

Our Microsoft Cloud revenue, which includes Azure and other cloud services, Office 365 Commercial, the commercial portion of LinkedIn, Dynamics 365, and other commercial cloud properties, was $137.4 billion, $111.6 billion, and $91.4 billion in fiscal years 2024, 2023, and 2022, respectively. These amounts are primarily included in Server products and cloud services, Office products and cloud services, LinkedIn, and Dynamics products and cloud services in the table above.
Distance: 0.2738
--------------------------------------------------
Highlights from fiscal year 2024 compared with fiscal year 2023 included: 

•	Microsoft Cloud revenue increased 23% to $137.4 billion. 

•	Office Commercial products and cloud services revenue increased 14% driven by Office 365 Commercial growth of 16%. 

•	Off

In [67]:
print(
    ask_report_v3(
        "What was Microsoft's total revenue in fiscal year 2024?",
        k=5,
        year=2024
    )[0]
)

print(
    ask_report_v3(
        "How much revenue did Microsoft Cloud generate in fiscal year 2024?",
        k=5,
        year=2024
    )[0]
)

print(
    ask_report_v3(
        "What was Microsoft's net income in fiscal year 2025?",
        k=5,
        year=2025
    )[0]
)

print(
    ask_report_v3(
        "What was Microsoft's operating income in fiscal year 2025?",
        k=5,
        year=2025
    )[0]
)

print(
    ask_report_v3(
        "What was Microsoft's total revenue in fiscal year 2023?",
        k=5,
        year=2023
    )[0]
)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


I could not find this information in the provided reports.


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


In fiscal year 2024, Microsoft Cloud revenue was $137.4 billion (Source: 2024_Annual_Report.docx, Year: 2024).


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Based on the 2025 annual report, Microsoft's net income in fiscal year 2025 was $101,832 million (or $101,832,000,000).


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Based on the 2025 annual report, Microsoft's operating income in fiscal year 2025 was $128,528 million (or $128,528,000,000).


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


In fiscal year 2023, Microsoft's total revenue was $211,915 million (or $211.9 billion).

Source: 2023_Annual_Report.docx (Year: 2023)


In [68]:
print(
    ask_report_v3(
        "What was Microsoft's operating income in fiscal year 2025?",
        k=5,
        year=2025
    )[0]
)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Based on the 2025 annual report, Microsoft's operating income in fiscal year 2025 was $128,528 million (or $128.528 billion).
